In [0]:
import data.utilities.common as Commonlib
import pyspark.sql.functions as F
import pyspark.sql.types as T
import yaml

from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.medallion_table_utility import MedallionTableUtility as MTU
from data.utilities.publishers import SnowflakeExternalTable

In [0]:
CONFIG_BASE_PATH = "configuration/"

config_selection_path = f"{CONFIG_BASE_PATH}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "configuration file")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}/{config_file}"

config = Commonlib.get_object_config(config_file_path)

In [0]:
# Instantiate medallion object
medallion = Medallion(config)

In [0]:
SILVER_TABLES = {
    "wslr_dir": MTU.get_fully_qualified_table_name("external", "stlpr205", "wslr_dir"),
    "wslr_dat": MTU.get_fully_qualified_table_name("external", "stlpr205", "wslr"),
}

# defining constants to reduce duplication
WSRL_DIR_TYPE = "wslr_dir.type"
WSLR_DAT_TYPE = "wslr_dat.wslr_type"
WSLR_CITY_ST_NM = "wslr_dir.wslr_city_st_nm"

In [0]:
# Transformations
wslr_addr_typ_cd = (
    F.when((F.col(WSRL_DIR_TYPE) == "1"), F.lit("SHIPTO"))
    .when((F.col(WSRL_DIR_TYPE) == "5"), F.lit("BILLTO"))
    .when((F.col(WSRL_DIR_TYPE) == "3"), F.lit("MAILTO"))
)

wslr_addr_1_txt = F.when((F.col("wslr_dir.wslr_addr_1_txt") == "NOT AVAILABLE"), F.lit(None)).otherwise(
    F.col("wslr_dir.wslr_addr_1_txt")
)

wslr_addr_2_txt = F.when((F.col("wslr_dir.wslr_addr_2_txt") == ""), F.lit(None)).otherwise(
    F.col("wslr_dir.wslr_addr_2_txt")
)

wslr_city_nm = F.when((F.col(WSLR_DAT_TYPE) == "EP"), F.col(WSLR_CITY_ST_NM)).otherwise(
    F.when((F.col(WSLR_CITY_ST_NM) == "NOT AVAILABLE"), F.lit(None)).otherwise(
        F.expr(
            "substring(wslr_dir.wslr_city_st_nm, 1, length(wslr_dir.wslr_city_st_nm) - instr(reverse(wslr_dir.wslr_city_st_nm), ' '))"
        )
    )
)

wslr_st_cd = F.when((F.col(WSLR_DAT_TYPE) == "EP"), F.col(WSLR_CITY_ST_NM)).otherwise(
    F.when((F.col(WSLR_CITY_ST_NM) == "NOT AVAILABLE"), F.lit(None)).otherwise(
        F.split(F.col(WSLR_CITY_ST_NM), " ")[F.size(F.split(F.col(WSLR_CITY_ST_NM), " ")) - 1]
    )
)

In [0]:
# mapping from silver tables to gold wholesaler_address table
wslr_addr_col_map = {
    "wslr_nbr": F.col("wslr_dir.wslr_nbr"),
    "wslr_addr_typ_cd": wslr_addr_typ_cd,
    "wslr_nm": F.col("wslr_dir.wslr_name"),
    "wslr_addr_1_txt": wslr_addr_1_txt,
    "wslr_addr_2_txt": wslr_addr_2_txt,
    "wslr_addr_3_txt": F.lit(None).cast(T.StringType()),
    "wslr_city_st_nm": F.col(WSLR_CITY_ST_NM),
    "wslr_city_nm": wslr_city_nm,
    "wslr_st_cd": wslr_st_cd,
    "wslr_postal_cd": F.col("wslr_dir.wslr_zip_code"),
}

In [0]:
try:
    # base df for join
    left_df = (
        spark.read.table(SILVER_TABLES["wslr_dir"]).alias("wslr_dir").filter(F.col(WSRL_DIR_TYPE).isin("1", "3", "5"))
    )
    # next silver df
    wslr_dat_df = (
        spark.read.table(SILVER_TABLES["wslr_dat"])
        .alias("wslr_dat")
        .select(F.col("wslr_dat.wslr_nbr").alias("wslr_dat_nbr"), F.col(WSLR_DAT_TYPE))
        .alias("wslr_dat")
    )
    medallion.logger.info("Reading in silver tables for wholesaler address procedure")

    # join silver tables
    left_df = left_df.alias("wslr_dir").join(
        other=wslr_dat_df, on=F.col("wslr_dir.wslr_nbr") == F.col("wslr_dat.wslr_dat_nbr"), how="left"
    )
    medallion.logger.info("Performing join operation for wholesaler address procedure")

    # Transform and select final columns
    medallion.df = (
        left_df.withColumns(wslr_addr_col_map)
        .select(*wslr_addr_col_map.keys())
        .withColumn("__created_tsp", F.lit(medallion.start_datetime).cast(T.TimestampType()))
    )
    medallion.logger.info("Transforming and selecting final columns for wholesaler address procedure")
except Exception as e:
    medallion.logger.error(f"Error processing wholesaler_address source table reads/joins. Error message: {e}")
    raise

In [0]:
# write gold table
try:
    medallion.write_to_delta(load_type="overwrite", layer="gold")
    medallion.logger.info("wholesaler address successfully written to gold layer")
except Exception as e:
    medallion.logger.error(f"Error writing wholesaler address table to gold layer. Error message: {e}")
    raise

In [0]:
# publish to external stage - snowflake
try:
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()
except Exception as e:
    medallion.logger.error(e)
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config["name"],
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
        config.get("comparison_check", {}).get("treat_nulls"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )